# OnKith Model V2 data preparation

This notebook audits the pinned data and demonstrates deterministic, span-aware preparation. It does **not** train DeBERTa or any other model.

## 1. Imports and fixed experiment settings

Keep the dataset revision and random seed visible. Offline mode prevents an unnoticed dataset update or download.

In [ ]:
import json
import os
from pathlib import Path

from augmentation import (
    add_conversational_frame,
    apply_case_variant,
    build_email_values,
    diversify_entity_values,
    generate_hard_negatives,
    punctuation_free,
    speak_structured_entities,
    validate_record,
)
from data_audit import DATASET_NAME, DATASET_REVISION, SEED, build_current_report
from grouped_splits import split_by_factor

ROOT = Path.cwd()
if not (ROOT / 'augmentation.py').is_file():
    ROOT = Path('model/privacy_data_v2').resolve()

os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
print('Dataset:', DATASET_NAME, '@', DATASET_REVISION)
print('Seed:', SEED)
print('Root:', ROOT)

## 2. Reproduce the current dataset audit

This loads the pinned cached dataset, reproduces the historical entity-presence split, and reports unsupported and rare labels. It preserves the official validation split as the IID test.

In [ ]:
audit_report = build_current_report()
split = audit_report['historical_split']
print(json.dumps(split, indent=2))
print('Project-train empty rows:', audit_report['project_train']['empty_mask_rows'])
print('Labels below 100 spans:', audit_report['project_train']['rare_labels']['fewer_than_100'])

## 3. Load curated resources

The values are synthetic or curated for surface diversity. Reserved example domains avoid creating live contact information.

In [ ]:
resources = ROOT / 'resources'
entity_values = json.loads((resources / 'entity_values.json').read_text(encoding='utf-8'))
email_parts = json.loads((resources / 'email_domains.json').read_text(encoding='utf-8'))
entity_values['EMAIL'] = build_email_values(email_parts['local_parts'], email_parts['domains'])
frames = json.loads((resources / 'conversational_frames.json').read_text(encoding='utf-8'))
hard_negative_templates = json.loads((resources / 'hard_negative_templates.json').read_text(encoding='utf-8'))
print('Entity inventories:', sorted(entity_values))
print('Conversational frames:', len(frames))
print('Hard-negative templates:', len(hard_negative_templates))

## 4. Demonstrate span-aware diversification

Use a normalized example here. Real dataset rows must first pass the existing strict annotation normalization; these utilities deliberately reject overlapping or stale spans.

In [ ]:
text = 'Email Ana at ana.silva@example.com.'
example = {
    'uid': 'notebook-example',
    'source_text': text,
    'privacy_mask': [
        {'label': 'GIVENNAME', 'start': 6, 'end': 9, 'value': 'Ana'},
        {'label': 'EMAIL', 'start': 13, 'end': 34, 'value': 'ana.silva@example.com'},
    ],
    'partition': 'train_candidate',
}

variants = [
    diversify_entity_values(example, entity_values, seed=SEED),
    apply_case_variant(example, 'lower'),
    punctuation_free(example),
    speak_structured_entities(example),
    add_conversational_frame(example, frames[0]['prefix'], frames[0]['suffix']),
]
for variant in variants:
    validate_record(variant)
    print(variant['augmentation_family'], '->', variant['source_text'])

## 5. Generate realistic hard negatives

These are non-private examples that resemble PII. They are candidates for training and development only, never substitutes for the frozen OOD-final suite.

In [ ]:
hard_negatives = generate_hard_negatives(hard_negative_templates, seed=SEED)
print('Generated hard negatives:', len(hard_negatives))
for row in hard_negatives[:5]:
    print(row['hard_negative_category'], '->', row['source_text'])

## 6. Audit one-factor grouped splits

A split declares exactly which factor it holds out. Connected components prevent that factor from crossing train and holdout, and the returned audit records any imbalance caused by large components.

In [ ]:
split_rows = variants + hard_negatives[:10]
train_indices, holdout_indices, split_audit = split_by_factor(
    split_rows,
    factor='canonical_template',
    holdout_fraction=0.20,
    seed=SEED,
)
print(json.dumps(split_audit, indent=2))
assert split_audit['is_disjoint']

## 7. Review before future training

Before materializing a training corpus, choose augmentation ratios using training/development data only, inspect generated text manually, verify group-overlap reports, and keep the frozen OOD-final fixtures inaccessible. Model training remains a later phase.